In [ ]:
class Transaccion:
    """Guarda la informacion de UN movimiento de dinero."""

    def __init__(self, tipo, monto, saldo_resultante):
        self.__tipo = tipo                         # "Deposito", "Retiro", etc.
        self.__monto = monto
        self.__saldo_resultante = saldo_resultante

    def get_tipo(self):
        return self.__tipo

    def get_monto(self):
        return self.__monto

    def get_saldo_resultante(self):
        return self.__saldo_resultante

    def __str__(self):
        return (f"{self.__tipo:<24} S/ {self.__monto:>10.2f}   "
                f"Saldo: S/ {self.__saldo_resultante:>10.2f}")


# =====================================================================
#                         CLASE PADRE: Cuenta
# =====================================================================

class Cuenta:
    """Clase PADRE: lo que tienen en comun todas las cuentas bancarias."""

    # Variable de CLASE: la comparten todas las cuentas.
    # Sirve para dar un numero de cuenta unico (igual que "tag" en Rabbit).
    contador = 1

    def __init__(self, titular, saldo_inicial=0):
        # Atributos PRIVADOS (encapsulamiento)
        self.__numero = "001-" + str(Cuenta.contador).zfill(5)   # ej: 001-00001
        Cuenta.contador += 1
        self.__titular = titular           # objeto Cliente (relacion entre clases)
        self.__saldo = 0
        self.__movimientos = []            # LISTA de objetos Transaccion
        if saldo_inicial > 0:
            self.depositar(saldo_inicial, "Apertura de cuenta")

    # ---------------- GETTERS ----------------
    def get_numero(self):
        return self.__numero

    def get_titular(self):
        return self.__titular

    def get_saldo(self):
        return self.__saldo

    def get_movimientos(self):
        return self.__movimientos

    def get_tipo(self):
        return "Cuenta"

    # NOTA: a proposito NO existe set_saldo().
    # El saldo solo puede cambiar con depositar() o retirar().

    # ---------------- METODOS ----------------
    def __registrar(self, tipo, monto):
        """Metodo PRIVADO: agrega un movimiento al historial."""
        self.__movimientos.append(Transaccion(tipo, monto, self.__saldo))

    def depositar(self, monto, tipo="Deposito"):      # argumento por defecto
        if monto <= 0:
            print("El monto a depositar debe ser mayor que 0.")
            return False
        self.__saldo += monto
        self.__registrar(tipo, monto)
        return True

    def puede_retirar(self, monto):
        """Regla general: no se puede retirar mas de lo que hay.
        Las subclases SOBRESCRIBEN este metodo con sus propias reglas."""
        return monto <= self.__saldo

    def retirar(self, monto, tipo="Retiro"):
        if monto <= 0:
            print("El monto a retirar debe ser mayor que 0.")
            return False
        if not self.puede_retirar(monto):     # usa la version de la subclase
            print(f"Operacion rechazada: no se puede retirar S/ {monto:.2f} "
                  f"de la cuenta {self.__numero}.")
            return False
        self.__saldo -= monto
        self.__registrar(tipo, monto)
        return True

    def cierre_de_mes(self):
        """Cada tipo de cuenta hace algo distinto a fin de mes (se sobrescribe)."""
        pass

    def estado_de_cuenta(self):
        print(f"\n===== ESTADO DE CUENTA {self.__numero} =====")
        print(f"Tipo    : {self.get_tipo()}")
        print(f"Titular : {self.__titular.get_nombre_completo()}")
        print("-" * 58)
        if len(self.__movimientos) == 0:
            print("  (sin movimientos)")
        for i in range(len(self.__movimientos)):
            print(f"  {i + 1:>2}. {self.__movimientos[i]}")
        print("-" * 58)
        print(f"  SALDO ACTUAL: S/ {self.__saldo:.2f}")

    def __str__(self):
        return (f"{self.__numero}  {self.get_tipo():<17} "
                f"Saldo: S/ {self.__saldo:>10.2f}")
